# Hidden Gems

This notebook explains. The calculation lives in `devkit/scoring.py`. You should be able to read it top to bottom without running anything, and every number here should be traceable to a function with a name.

## Table of contents

1. [What this is, and what it isn't](#1-what-this-is-and-what-it-isnt)
2. [What changed since `demo_dashboard1`](#2-what-changed-since-demo_dashboard1)
3. [New files added](#3-new-files-added)
4. [Setup](#4-setup)
    - 4.1. [The header contract](#41-the-header-contract)
    - 4.2. [The league map](#42-the-league-map)
5. [How a score gets built](#5-how-a-score-gets-built)
    - 5.1. [Two score columns, answering different questions](#51-two-score-columns-answering-different-questions)
6. [Answers to the open TODOs from `demo_dashboard1`](#6-answers-to-the-open-todos-from-demo_dashboard1)
    - 6.1. ["check if it's justified that these assumptions are completely wrong or if I've messed up my rationale"](#61-check-if-its-justified-that-these-assumptions-are-completely-wrong-or-if-ive-messed-up-my-rationale)
    - 6.2. ["linear and equal weights is not conceptually correct"](#62-linear-and-equal-weights-is-not-conceptually-correct)
    - 6.3. ["Role based is giving issues due to underrepresentation"](#63-role-based-is-giving-issues-due-to-underrepresentation)
7. [What we still don't know](#7-what-we-still-dont-know)
8. [Open questions/Doubts](#8-open-questionsdoubts)

### 1. What this is, and what it isn't
[[go back to the top]](#table-of-contents)

It is a deterministic filter. It narrows roughly 40,000 players down to a candidate list small enough to look at by hand, or to hand to a model trained to spot potential. It exists because we have no access to bulk data and cannot pull everything manually.

It is not a valuation model. It does not try to reproduce market value and should not be judged on that. **The goal is potential, not price.** A 30-year-old worth €5 million and a 17-year-old with no market value are judged using exactly the same criteria: **how well they perform for their role and league, right now**. `Market value` never enters the score, in either direction, it is looked up afterwards, next to the score, purely as a reference label, never as an input.

The pool does include a lot of players without a market value, but that’s simply a result of the data we’re working with, not something the filter is deliberately looking for. In fact, every player aged 16–19 in the dataset has no market value yet, as shown in the age/price table further down.

So naturally, the candidate list ends up containing many unpriced players. That’s a consequence of the dataset, not a goal of the selection process.

What makes a filter good is retention: **keeping the interesting players while cutting the search space**. That is the standard to hold it to, not agreement with price.

### 2. What changed since `demo_dashboard1` 
[[go back to the top]](#table-of-contents)

—> Read this Rafa and see if u agree.

`demo_dashboard1` scored every attacker off one flat list of stats, averaged with equal weight, comparing everyone against everyone across every league at once. Testing that against real market-value data turned up four concrete problems:

1. **Raw stats don't mean the same thing in every league, and the old score couldn't tell the difference.** A weaker league has weaker defenders and goalkeepers, so the same player racks up bigger raw numbers there than he would in a tougher league, more goals per 90, more successful dribbles, inflated by weak opposition, not necessarily by being a better player. This is *not* a claim that a weak-league standout should never beat a strong-league regular, sometimes he genuinely is better, and he still can under the new pipeline too (see step 6 below). The point is the old score compared raw numbers globally with no way to separate "genuinely excellent" from "inflated by a weak league", so nobody could trust the ranking in either direction.
2. **Minutes and age were scored as if they were skills.** Being young or having played a lot pushed the number up on its own, on top of whatever the player actually did on the pitch.
3. **`Market value = 0` was read as "worth nothing".** In Wyscout's export, 0 means "not priced", not "priced at zero". Since every young player is unpriced, this silently rewarded youth for the wrong reason.
4. **Only attacking mid/wingers/strikers had their own metrics.** Everyone else, which means that goalkeepers, defenders, deeper midfielders were scored on attacker vocabulary that doesn't describe their job at all.

Version 2 (this notebook, backed by `devkit/scoring.py`) fixes each one directly:

| | Before | Now |
|---|---|---|
| Percentiles | across all 75 leagues at once | within league and role |
| League level | ignored | coefficient applied at the end, from `league_map` |
| Minutes | a skill worth 10% | eligibility filter plus regularisation |
| Age | a skill worth 10% | its own column, outside performance |
| Percentages | 100% off one dribble scored as 100% | shrunk according to volume |
| Roles | attacking group only | all 8 roles |
| CSV header | assumed | validated against `docs/reference-header.txt` |
| `Market value` of 0 | treated as a price | treated as missing |

If there’s one thing to take from this table, it’s that the score itself never uses `Market value`, not before these changes, and not after them.

Everything we’ve done is about making the performance score reliable on its own, comparable across leagues and positions, without giving an unfair boost to younger players or those who simply play more minutes. Whether a player already has a market value is kept completely separate and shown alongside the score, purely as context. It never becomes part of the score.

On point 1 specifically: the fix is two steps, not one. First, every player is percentiled **within his own league and role** (step 2 below), that part is always fair, weak league or strong. Only *after* that does league strength (step 6) get blended in, at 35% weight. So a truly dominant player in a weak league (say, top 1% inside that league) can still outrank a merely-good player in a strong league, he just doesn't get the raw-stat inflation for free anymore.

### 3. New files added
[[go back to the top]](#table-of-contents)

It's worth knowing what each file added before reading further, because this notebook only *calls* these files , it doesn't contain the logic itself.

**`devkit/scoring.py`** — the calculator. Everything under "How a score gets built" below is implemented here. If you want to change how a score is computed, this is the file to edit. Can also be run directly from a terminal for a quick look: `python scoring.py --role CF --top 20`.

**`devkit/make_league_map.py`** → writes **`data/league_map.csv`** — the two go together. The script figures out, for every league folder under `data/`, a `strength` number (0-100) that says how strong that league is, so a player in Andorra and a player in the Serie B can be compared fairly. It writes its answer to the CSV. `scoring.py` just reads that CSV, it does not recompute anything. Re-run the script (`python make_league_map.py`) only if a new league folder gets added, or if the strength numbers themselves need revisiting, see the "Open questions" list about how provisional they still are.

One row per league folder (82 rows), these are the columns that matter day to day:

| Column | What it is |
|---|---|
| `league_code` | short stable key used in code, e.g. `GER_22B` for the 2. Bundesliga |
| `country`, `league` | the country code and competition name, read straight from the folder name |
| `division` | national tier, 1 = top flight, 2 = second tier, and so on |
| `priority`, `priority_rank` | the Dribblify scouting colour (green / yellow / red) —> **where** it's worth looking, not how strong the league is |
| **`strength`** | **the number `scoring.py` actually uses** —> 0 to 100, how strong the league is, on the same scale for every country |
| `strength_source` | where that `strength` number came from, see the three values just below |
| `n_teams`, `n_players` | counted straight from that league's CSV, mostly a sanity check |
| `calendar` | autumn-spring / calendar-year / apertura-clausura —> when the season runs, matters for lining up future seasons |
| `notes` | anything the pipeline shouldn't silently assume, e.g. "no CSV exported yet" |

`strength_source` tells you which of three routes produced the `strength` number:
- **`provisional_uefa_x_division`** (48 leagues) —> European country: the UEFA association coefficient for that country, multiplied by how much of the top-flight level a lower division keeps (100% for the 1st tier, 55% for the 2nd, 32% for the 3rd, 18% for the 4th , a hand-set assumption, not derived from data, see the open questions).
- **`provisional_marketvalue_interp (coverage X%)`** (30 leagues) —> outside Europe, no UEFA coefficient exists: uses that league's median `Market value` instead, converted onto the same 0-100 scale as the UEFA route. `coverage` is what share of that league's players actually have a price behind that median, low coverage means fewer players to trust the median.
- **`MISSING - fill by hand`** (4 leagues) —> neither route worked, because there's no CSV exported for that league yet (MLS, USL Championship, South Africa's Premier Division, Algeria's Ligue 1) and none of them are UEFA members either. These four still get scored fairly within their own league, they just can't be compared against any other league yet.

**`docs/reference-header.txt`** — the list of 115 column names every league CSV is expected to have, one per line. `scoring.py` checks every file against this list before using it (see "The header contract" below) and skips anything that doesn't match, instead of silently loading a broken file. If Wyscout's export format ever changes, this is the file to regenerate.

**`devkit/concept_analysis.py`** — not part of the pipeline, a verification tool. It exists to answer one narrow question honestly: are the numbers quoted in "Answers to the open TODOs" (further down) actually measured, or just typed? Run it (`python concept_analysis.py`) and it recomputes every one of those numbers from the current data. If it and the notebook ever disagree, trust the script and re-copy its output into the notebook, not the other way around.

**`reports/ale_sample/potential_index.py`** — builds the company-facing dashboard. It calls `scoring.build()`, reshapes the result into the JSON format the HTML template expects, and writes `potential_dashboard.html`. **This is the file to hand to the company**, it needs nothing installed, just open it in a browser. `dashboard_template.html` (also in that folder) is the empty shell with all the layout and charts; `potential_index.py` is what fills it with real players. Re-run `python potential_index.py` after any change to `scoring.py` to refresh it.


### 4. Setup
[[go back to the top]](#table-of-contents)


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "devkit"))

import pandas as pd
import scoring

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

#### 4.1. The header contract
[[go back to the topic]](#4-setup)

Every CSV is checked against `docs/reference-header.txt`, the 115 canonical columns. A different Wyscout preset produces missing columns that would flow through the percentiles as `NaN` and quietly turn into bottom of the table scores. That is the failure this check refuses to let through.

In [2]:
ref = scoring.reference_header()
print(f"{len(ref)} columns under contract")
print(ref[:6], "...")

115 columns under contract
['Player', 'Team', 'Team within selected timeframe', 'Position', 'Age', 'Market value'] ...


#### 4.2. The league map
[[go back to the topic]](#4-setup)

The CSVs know nothing about the competition they belong to beyond the folder name. The 2. Bundesliga and the Andorran first division both sit in `data/2025-2026`, full of players putting up similar numbers at completely different levels.

There is a distinction here that already caught us out once. The green, yellow and red bands in the Dribblify map are scouting priority, not league strength. The Campeonato de Portugal, a fourth tier competition, is green. The Serie B is yellow. Using priority as a level coefficient would rank the Campeonato de Portugal above the Serie B, despite Serie B is better in terms of playr quality, so `priority` and `strength` are kept as separate columns.

The `strength` values are provisional, and `strength_source` records where each one came from.

In [3]:
lm = scoring.load_league_map()
print(lm["strength_source"].str.split(" ").str[0].value_counts().to_string())
lm.dropna(subset=["strength"]).nlargest(8, "strength")[
    ["league_code", "league", "priority", "division", "strength", "strength_source"]]

note: 4 league(s) have no strength yet - their players are scored within their own league but left out of cross-league comparison
strength_source
provisional_uefa_x_division       48
provisional_marketvalue_interp    30
MISSING                            4


,league_code,league,priority,division,strength,strength_source
16,NED_EREDI,Eredivisie,green,1,100.0,provisional_uefa_x_division
48,MEX_LM,Liga MX,yellow,1,100.0,provisional_marketvalue_interp (coverage 58%)
26,BEL_PL,Pro League,yellow,1,91.3,provisional_uefa_x_division
44,ITA_SB,Serie B,yellow,2,83.4,provisional_uefa_x_division
0,GER_22B,2. Bundesliga,green,2,76.3,provisional_uefa_x_division
24,AUT_BUNDE,Bundesliga,yellow,1,70.8,provisional_uefa_x_division
46,JPN_J1L,J1 League,yellow,1,70.8,provisional_marketvalue_interp (coverage 70%)
55,TUR_SPL,Süper Lig,yellow,1,70.5,provisional_uefa_x_division


### 5. How a score gets built
[[go back to the top]](#table-of-contents)

Seven steps, in this order. The order matters, because the league coefficient only enters after the percentiles have already been computed inside each league.

1. **Eligibility.** 600 minutes and a known role.
2. **Role metrics.** Percentiled within league and role, so the level of a league does not get baked into the number itself.
3. **Small cells.** A league and role group with fewer than 10 players gets compared against the same role pooled across leagues of similar strength.
4. **Thin percentages.** A percentage built on few attempts is shrunk toward the role average, in proportion to volume.
5. **Short seasons.** The score is shrunk by the square root of minutes over 1800. Half a season is evidence, not proof.
6. **League level.** Only now does the coefficient enter, to make leagues comparable.
7. **Age.** Kept out of the performance score entirely, in a column of its own.

Age and minutes are not skills. Mixing "how young he is" into "how well he played" makes both unreadable, and it left the age term correlating at &minus;0.20 with the very index it was part of.

In [4]:
res = scoring.build()

print(f"{len(res)} eligible players | {res['league_code'].nunique()} leagues | {res['role'].nunique()} roles")
print(f"{int(res['priced'].sum())} priced, {int((~res['priced']).sum())} unpriced")
res.groupby("role_label").agg(
    players=("Player", "size"),
    mean_age=("age", "mean"),
    mean_minutes=("minutes", "mean"),
).round(0)

note: 4 league(s) have no strength yet - their players are scored within their own league but left out of cross-league comparison
note: 215 player(s) in league x role cells under 10 - percentiled against their strength band instead


24873 eligible players | 75 leagues | 8 roles
14488 priced, 10385 unpriced


,players,mean_age,mean_minutes
role_label,,,
Attacking midfielder,1088,27.0,1602.0
Central midfielder,3120,27.0,1724.0
Centre back,4962,27.0,1875.0
Centre forward,3788,27.0,1508.0
Defensive midfielder,1922,27.0,1736.0
Full back,4640,26.0,1732.0
Goalkeeper,1908,28.0,2055.0
Winger,3445,26.0,1508.0


#### 5.1. Two score columns, answering different questions
[[go back to the topic]](#5-how-a-score-gets-built)

`perf_league` says how good a player is inside his own league and role. That is the right question when you are trying to find someone.

`perf_adjusted` applies the league level coefficient on top. That is the right question when you need to compare someone in Andorra against someone in the Serie B.

They are not interchangeable, and the list changes depending on which one you sort by.

In [5]:
scoring.shortlist(res, role="CF", max_age=23, top=12)

,Player,club_season,league,age,minutes,market_value,perf_league,perf_adjusted,evidence
0,R. Pepi,PSV,Eredivisie,23,1514,NaN,81.4,87.9,0.917
1,N. Tresoldi,Club Brugge,Pro League,21,2680,5000000.0,85.8,87.7,1.000
2,M. Fall,La Louvière,Pro League,22,1964,NaN,84.1,86.6,1.000
3,A. González,Guadalajara,Liga MX,23,2723,NaN,75.7,84.2,1.000
4,Nacho Ferri,Westerlo,Pro League,21,3064,NaN,79.9,83.9,1.000
5,M. Zonneveld,Sparta Rotterdam,Eredivisie,22,1244,NaN,72.7,82.3,0.831
6,R. Vermant,Club Brugge,Pro League,22,1510,1800000.0,75.5,81.0,0.916
7,S. Shpendi,Empoli,Serie B,23,2729,2000000.0,76.7,79.0,1.000
8,O. Diakité,Cercle Brugge,Pro League,22,1566,NaN,70.8,78.0,0.933
9,M. Cvetković,Anderlecht,Pro League,19,1802,NaN,70.0,77.5,1.000


### 6. Answers to the open TODOs from `demo_dashboard1`
[[go back to the top]](#table-of-contents)


#### 6.1. "check if it's justified that these assumptions are completely wrong or if I've messed up my rationale"
[[go back to the topic]](#6-answers-to-the-open-todos-from-demo_dashboard1)

The rationale was not wrong. The validation target was.

Nothing below is typed in. Every number is computed live in this cell by calling `devkit/concept_analysis.py`, on the original flat 10-concept index from `demo_dashboard1` (attacking group only, AM/W/CF). If you want to poke at it yourself later, that script is the place.

**Everywhere in 6.1, 6.2 and 6.3, "index" means this old `demo_dashboard1` index**, the `index` variable computed in the next cell. It has nothing to do with `perf_league` / `perf_adjusted`, the current pipeline's real score from [[section 5.]](#5-how-a-score-gets-built). This old index only exists here, to test the old TODOs against real numbers. Nobody ranks anyone with it.

In [6]:
import concept_analysis as ca

att = ca.load_attackers()
mv = pd.to_numeric(att["Market value"], errors="coerce")
scores = ca.concept_scores(att)
index = ca.flat_index(scores)

baseline = index.corr(mv, method="spearman")
no_age = ca.flat_index(scores.drop(columns=["Age (younger)"])).corr(mv, method="spearman")
within_league = ca.flat_index(ca.concept_scores(att, within_league=True)).corr(mv, method="spearman")

print(f"n attackers = {len(att)}\n")
print(f"baseline (10-concept index) vs Market value : {baseline:.2f}")
print(f"drop the age term                            : {no_age:.2f}")
print(f"percentile within league, not globally       : {within_league:.2f}")

C:\Users\aleja\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.4.2)
  from scipy.sparse import csr_matrix, issparse


n attackers = 15012

baseline (10-concept index) vs Market value : 0.18
drop the age term                            : 0.28
percentile within league, not globally       : 0.15


Dropping age increases the correlation from 0.18 to 0.28. That makes sense because the index rewards younger players, while market value tends to reward players who are already established. Percentiling within the league has the opposite effect, bringing the correlation down from 0.18 to 0.15. This is also useful because it shows that part of the original relationship with market value was actually coming from which league the player plays in, rather than how well he performed. Percentiling within each league removes that effect on purpose.

Neither of these changes explains why the overall correlation is still weak. The main issue is that `Market value` isn’t a reliable ground truth for this comparison, for three separate reasons that we calculate below.


In [7]:
# reason 1: the target population has no price
band = pd.cut(res["age"], bins=[0, 19, 21, 25, 200], labels=["16-19", "20-21", "22-25", "26+"])
print("share with NO market value, by age band:")
print(((~res["priced"]).groupby(band, observed=True).mean() * 100).round(0).to_string())
print(f"\nof priced players, share aged 26+: {(res.loc[res['priced'], 'age'] >= 26).mean()*100:.0f}%\n")

# reason 2: price is mostly just the league
res["league_median_mv"] = res.groupby("league_code")["market_value"].transform("median")
r = res.loc[res["priced"], "league_median_mv"].corr(res.loc[res["priced"], "market_value"])
print(f"corr(own price, own league's median price) = {r:.2f}   (our index only managed {baseline:.2f})\n")

# reason 3: price has almost no resolution
priced_mv = res.loc[res["priced"], "market_value"]
top6 = priced_mv.value_counts().nlargest(6)
print(f"distinct market values among {len(priced_mv)} priced players: {priced_mv.nunique()}")
print(f"share of priced players sharing just the 6 most common values: {top6.sum()/len(priced_mv)*100:.0f}%")

share with NO market value, by age band:
age
16-19    100.0
20-21     95.0
22-25     58.0
26+       21.0

of priced players, share aged 26+: 77%

corr(own price, own league's median price) = 0.42   (our index only managed 0.18)

distinct market values among 14488 priced players: 85
share of priced players sharing just the 6 most common values: 42%


**1. The target population has no price.** Every 16-19 year old in this base has no market value at all, and of the players who *do* carry a value, most are 26 or older. A correlation computed over the whole pool is measuring mostly players who are not the target.

**2. Price is mostly just the league.** A player's own price correlates more with his *league's* median price than with our performance index. That doesn't mean the index is worse than a constant. It means market value in these leagues is measuring largely which competition you play in, which is exactly why it's a poor target for an index meant to measure how you played.

**3. Price has almost no resolution.** Very few distinct price values are spread across thousands of priced players, and a big share of them share just six round numbers (€200k, €300k, €250k, €150k, €400k, €500k). That's not a valuation, it's a set of buckets.

On the second original question, why players worth 0 were ranking so highly: because 0 does not mean "worth nothing", it means "not priced". Since every player under 20 is unpriced and the index rewarded youth, it was selecting precisely the population with no price. This is fixed now, and a `Market value` of 0 reads as missing.

#### 6.2. "linear and equal weights is not conceptually correct"
[[go back to the topic]](#6-answers-to-the-open-todos-from-demo_dashboard1)

Confirmed, and now actually measured, using the same `att` / `scores` / `index` computed above in 6.1. No new setup needed.

The question: the old index gave each of its 10 concepts (goal output, chance creation, dribbling, minutes...) an equal 10% weight. Is that fair, or are some of those 10 secretly measuring the same thing?

In [8]:
# how much do the 10 concepts overlap with each other?
corr = scores.corr()
pairs = [("Chance creation", "Key passing"), ("Goal output", "Shot threat"),
         ("Chance creation", "Minutes"), ("Chance creation", "Dribbling")]
print("a few pairs that shouldn't be independent if the equal-weight assumption held:")
for a, b in pairs:
    print(f"  {a:17s} & {b:15s} : {corr.loc[a, b]:+.2f}")

a few pairs that shouldn't be independent if the equal-weight assumption held:
  Chance creation   & Key passing     : +0.73
  Goal output       & Shot threat     : +0.65
  Chance creation   & Minutes         : +0.45
  Chance creation   & Dribbling       : +0.35


They're not independent. The creative ones especially (chance creation, key passing) move together strongly. A player who creates chances tends to have a high number of key passes almost by definition, so counting both at 10% each is quietly giving "creativity" closer to 20%. Two more views of the same problem below: how many *actual* independent dimensions are the 10 concepts, and how much does each one really drive the final score once you measure it instead of assuming it.

In [9]:
from sklearn.decomposition import PCA

# how many independent dimensions are the 10 concepts really?
X = scores.fillna(scores.mean())
X = (X - X.mean()) / X.std()
ev = PCA().fit(X).explained_variance_ratio_
print(f"PC1 alone explains {ev[0]*100:.0f}% of the variance; the first 3 axes explain {ev[:3].sum()*100:.0f}%")
print("(if the 10 concepts were truly independent, each axis would explain about 10%)\n")

# how much does each concept actually drive the final index, vs its nominal 10%?
print("concept vs the final index it feeds into, sorted:")
print(scores.corrwith(index).sort_values(ascending=False).round(2).to_string())

PC1 alone explains 29% of the variance; the first 3 axes explain 58%
(if the 10 concepts were truly independent, each axis would explain about 10%)

concept vs the final index it feeds into, sorted:
Minutes            0.72
Chance creation    0.71
Key passing        0.67
Dribbling          0.57
Through balls      0.56
Shot threat        0.50
Goal output        0.43
Passing            0.42
Fouls drawn        0.33
Age (younger)     -0.20


The 10 concepts behave more like 3 real dimensions than 10. And the "actual weight" table shows the real spread: `Age (younger)` barely moves the index (and pulls the wrong way), while `Minutes` and `Chance creation` dominate it, nowhere close to the intended 10% each.

**So why not just fix it?** We tried grouping the 10 concepts into fewer, non-overlapping blocks, which is the conceptually correct move, and checked what it did to *retention*: of the top 15% of players by market value, how many does the index's own top 15% actually recover.

**NOTE:** [[Section 1.]](#1-what-this-is-and-what-it-isnt) says the goal is potential, not price, and that agreement with price is explicitly not the standard to judge this by. So why measure "retention" against `Market value` here at all? Because we have no direct way to measure "potential", nobody has a list of "these players truly had potential" to check against. `Market value` is not what we're trying to predict, it's just the only real-world number we have anywhere close to player quality, used here as a rough smoke test, not as the actual goal. It's genuinely useful for catching outright bugs (like the `Market value = 0` mistake in 6.1). It's a bad tool for judging *this* specific fix, tho price rewards the exact redundancy we're trying to remove, so a drop in this "retention" number does not mean the fix is bad, it means this ruler cannot tell us either way. The test that would actually answer "does this catch potential" is the retrospective transfer test in [[section 7.]](#7-what-we-still-dont-know), not this one.

In [10]:
# does grouping the redundant concepts together actually help?
blocks_7 = {
    "Goal threat": ["Goal output", "Shot threat"],
    "Creative": ["Chance creation", "Through balls", "Key passing"],
    "Dribbling": ["Dribbling"], "Passing": ["Passing"],
    "Fouls drawn": ["Fouls drawn"], "Minutes": ["Minutes"],
    "Age (younger)": ["Age (younger)"],
}
block_scores = pd.DataFrame({name: scores[cols].mean(axis=1) for name, cols in blocks_7.items()})
index_7block = ca.flat_index(block_scores)

print(f"retention, 10 concepts (current)     : {ca.retention(index, mv):.0%}")
print(f"retention, 7 blocks (de-duplicated)  : {ca.retention(index_7block, mv):.0%}")

retention, 10 concepts (current)     : 23%
retention, 7 blocks (de-duplicated)  : 19%


Retention got *worse* after de-duplicating, not better. The price proxy rewards exactly the redundancy that the correct fix removes, so the aggregation can't be tuned against market value, because that would optimise for reproducing the bias in the price rather than removing it. Until there's a better ground truth than price, this stays a conceptual choice, not an empirical one.

**This is not fully fixed in `scoring.py` either.** `ROLE_METRICS` is a shorter, role-specific list than the old flat 10 concepts, which helps, but it still has the same kind of overlap in places. For example, full-backs and wingers are scored on both `Crosses per 90` and `Accurate crosses, %`, volume and accuracy of the same underlying skill counted twice. It's a known, accepted gap, not an oversight, for the same reason as above: there's no trustworthy target to tune the de-duplication against yet.

#### 6.3. "Role based is giving issues due to underrepresentation"
[[go back to the topic]](#6-answers-to-the-open-todos-from-demo_dashboard1)

Correct, and now measured live too, same `att` / `index` from 6.1. Under the old shared concept list, how did the three attacking roles (centre forward, winger, attacking midfielder) split among the priced players, and among the top of the ranking?

In [11]:
# role split among priced players, vs among the top 1000 by the old flat index
# (a market value of 0 means "not priced", same rule as everywhere else in this notebook)
priced_mask = mv.notna() & (mv > 0)
priced_split = att.loc[priced_mask, "Role"].value_counts(normalize=True).mul(100).round(0)
top1000_split = att.loc[index.nlargest(1000).index, "Role"].value_counts(normalize=True).mul(100).round(0)

print(f"priced universe (n={int(priced_mask.sum())}), role split:")
print(priced_split.to_string())
print(f"\ntop 1000 by the old index, role split:")
print(top1000_split.to_string())

priced universe (n=6504), role split:
Role
CF    47.0
W     39.0
AM    14.0

top 1000 by the old index, role split:
Role
W     60.0
AM    22.0
CF    19.0


Wingers tripled their share while centre forwards fell to less than half, because the shared list is dominated by dribbling, progressive carries, key passes and fouls drawn. That is winger vocabulary. It was the underrepresentation problem happening inside the attacking group, which had been picked precisely because it looked well represented.

Step 2 fixes it. Each role is now scored on its own metrics, using the `ROLE_METRICS` that was already defined but never called, and percentiled inside its own group. A centre back no longer competes with a striker's shooting volume, and the cell below shows the current role counts under that fixed pipeline.

In [12]:
res.groupby("role_label").size().sort_values(ascending=False).rename("eligible").to_frame()

,eligible
role_label,
Centre back,4962
Full back,4640
Centre forward,3788
Winger,3445
Central midfielder,3120
Defensive midfielder,1922
Goalkeeper,1908
Attacking midfielder,1088


### 7. What we still don't know
[[go back to the top]](#table-of-contents)

Two things, stated plainly, because they are going to come up. By "the filter" in both, we mean the current pipeline, `scoring.py`, the `perf_league` / `perf_adjusted` numbers from [[section 5]](#5-how-a-score-gets-built). Not the old `demo_dashboard1` index that section 6 just used for diagnostics, that one only exists to check old claims, nobody uses it to actually rank anyone anymore.

**We don't actually know if the filter works, because we have never tested it.** Here's what "working" would mean in practice. We rank every player today. If that ranking is any good, the players near the top should be the ones who later get noticed, moved to a bigger club, picked up by scouts, something like that. We haven't checked this. Not "checked it and it was inconclusive". Checked, full stop, never happened.

To test it properly we'd need to rank players using old data, say the 2024/25 season, and then look at what actually happened to them afterwards, whether they moved to a bigger club in summer 2025. Doing it this way matters because the ranking has to be fixed before we know the outcome, otherwise we could end up unconsciously picking the examples that make the filter look good after the fact. Right now we can't run this test because we only have full 2024/25 data for two leagues (Segunda División RFEF and National 1), nowhere near enough to draw a real conclusion. This is the same decision as question 3 in the next section: **how much old data is worth collecting by hand so we can finally run this test**.

An earlier draft of this section stated specific transfer-rate numbers here (top-of-index players changing clubs more often than the rest, with p-values). Those numbers were not backed by any computation anywhere in this repository and have been removed. They should never have been written as if measured.

**The league strength numbers are first estimates, not something carefully validated yet.** They come from the UEFA association coefficient for European leagues, and from median market value for the rest of the world, as explained in [[section 3]](#3-new-files-added) above. Neither approach is perfect. Market-value coverage varies a lot between leagues, from just 2% to as much as 85%. In leagues with very little coverage, the players who do have a market value tend to be the more notable ones, which can make the league look stronger than it actually is.

We also thought about a third way to estimate strength: **look at players who played in more than one league, and see how their numbers changed when they moved**. We decided not to use it, because the players who change leagues are not a random sample. They're the ones who stood out enough to **get picked**, so their numbers before the move already carry some of that quality, not just the level of the league they were in. Comparing the two seasons directly would make almost any move look like a drop, even between two leagues of the exact same level. It's still a useful way to sanity-check strength numbers built some other way, just not a way to produce them from scratch.

### 8. Open questions/Doubts
[[go back to the top]](#table-of-contents)

Things worth a real conversation before the next round of work, not silent decisions.

1. **The goal is potential, not "cheap" or "unpriced".** This notebook used to read (my mistake) as if the target population was specifically players with no market value. That was wrong and has been corrected throughout. The score never uses `Market value`, so a priced player and an unpriced one compete on equal footing. If either of us has been mentally treating "no price" as the success condition, worth saying out loud, because it would quietly bias which candidates get looked at by hand.

2. **The redundant metrics inside `ROLE_METRICS` are still there** (same underlying problem as [[section 6.2.]](#62-linear-and-equal-weights-is-not-conceptually-correct) above, just at a smaller scale and not yet measured on the current pipeline). Full-backs and wingers are scored on both `Crosses per 90` and `Accurate crosses, %`, the same skill counted twice. Centre-forwards get three finishing-flavoured metrics (`Non-penalty goals per 90`, `xG per 90`, `Goal conversion, %`). Attacking midfielders get both `Dribbles per 90` and `Successful dribbles, %`. [[Section 6.2.]](#62-linear-and-equal-weights-is-not-conceptually-correct) already tried the obvious fix on the old index, grouping the redundant concepts into fewer blocks, and it made things worse against the price proxy, not better. So this isn't a simple "just fix it", the real decision is leave it as it is (current default, documented as a conscious gap), fix it anyway on the belief that it's conceptually right even without a way to test it, or hold off and put the effort into a better validation signal first (question 3's retrospective test could double as that, once the data exists) before touching any weights.

3. **We still haven't tested whether the filter actually works** (same point as [[section 7]](#7-what-we-still-dont-know) above, here as a decision to make rather than a fact). Running that test needs 2024/25 season data for at least the priority (green) leagues, matched against what actually happened to those players in summer 2025. Collecting a full season of data by hand for even one league takes real time, so worth agreeing as a team on how many leagues' worth of that effort is worth spending before this can say anything.

4. **Four leagues have no `strength` value at all** (`strength_source == MISSING`), see [[section 3]](#3-new-files-added) above. They still get scored fairly within their own league, but drop out of any cross-league comparison (`perf_adjusted`, `rank_overall`). Worth deciding who fills these in and from what source.

5. **Should `Market value` become a real target later, but only for a separate model, not for this filter?** This filter is deliberately blind to price and should stay that way. It exists to cut 40 000 down to a shortlist. A different, later-stage model, trained on the shortlist or on players once they do get priced, could try to predict market value or resale value as its own output. That's a downstream question, not a reason to let price back into this filter's scoring.

6. **Is this notebook meant for the company, or for us?** It carries real statistics (correlations, PCA, p-values) that assume a statistics background. For a football-side audience, the polished dashboard (`reports/ale_sample/potential_dashboard.html`) is probably much more appropriate to present live since it has the player names, filters, and keeps the technical details out of the way. This notebook is closer to an engineering log, useful for onboarding a teammate, understanding how the model works, or answering more detailed questions about the methodology. It’s probably not what we want to show in the meeting itself. Worth agreeing on that split explicitly.